# Data Pipeline

## Imports

In [1]:
import os
import json
from pathlib import Path
import pymupdf4llm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import AzureOpenAI
from FlagEmbedding import BGEM3FlagModel
from qdrant_client import QdrantClient, models
from dotenv import load_dotenv

c:\Users\raad\Desktop\GenAI-Pinnacle-Program\25_Capstone\dev\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Environment Variables

In [2]:
load_dotenv()

True

## Helper Functions

In [3]:
def get_reference(text, deployment = "gpt-5.4", api_version = "2024-12-01-preview"):

    sys_prompt = """You are an AI assistant that extracts the bibliographic reference of a scientific paper itself by analyzing its first page. 
Your task is to identify the paper’s own citation details and return them in JSON format with the following fields:

{
  "reference": {
    "title": "string or N/A",
    "first_author": "string or N/A",
    "year": "string or N/A",
    "publication": "string or N/A"
  }
}

Rules:
- Always output valid JSON only, with no extra commentary.
- "title" is the title of the paper.
- "first_author" is the first listed author of the paper.
- "year" is the publication year of the paper.
- "publication" is the journal, conference, or book where the paper was published.
- If any field cannot be found on the first page, set its value to "N/A".
- Do not attempt to extract references cited by the paper; only extract the reference of the paper itself.
"""

    client = AzureOpenAI(
        api_version=api_version,
        azure_endpoint=os.environ["AZURE_ENDPOINT"],
        api_key=os.environ["AZURE_API_KEY"],
    )

    response = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": sys_prompt,
            },
            {
                "role": "user",
                "content": text,
            }
        ],
        max_completion_tokens=2048,
        model=deployment
    )

    return json.loads(response.choices[0].message.content)

def format_reference(reference):
    _r = reference["reference"]
    return f"{_r['title']}, {_r['first_author']}, {_r['year']}"

def create_chunks(documents, references):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2500,    # Max characters per chunk, in average an english word is 5 characters
        chunk_overlap=0,  # Characters to repeat between chunks to keep context
        length_function=len,
        is_separator_regex=False,
    )

    chunks = []

    for doc, ref in zip(documents, references):
        ref_string = format_reference(ref)
        for page in doc:
            page_ref_string = ref_string + f", page {page['metadata']['page_number']}."
            _chunks = text_splitter.split_text(page["text"])
            _chunks = [f"Source: {page_ref_string}\n---\n{c}" for c in _chunks]
            chunks.extend(_chunks)
    
    return chunks

In [4]:
def prepare_pdf_chunks(file_dir):
    
    documents = []
    references = []
    
    for file in Path(file_dir).glob("*.pdf"):
        print(f"Parsing file: {file}")
        pages = pymupdf4llm.to_markdown(
            file,
            force_ocr=True,
            ocr_dpi=300,
            dpi=300,
            page_chunks=True,
            table_strategy="lines_strict",
            ocr_language="eng",
            header=False,
            footer=False,
        )
        documents.append(pages)

    for document in documents:
        reference = get_reference(document[0]["text"])
        references.append(reference)
    
    chunks = create_chunks(documents, references)

    return chunks


In [6]:
file_dir = "../../data"

chunks = prepare_pdf_chunks(file_dir)

Parsing file: ..\..\data\attention_paper.pdf
=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=1/2.
OCR on page.number=2/3.
OCR on page.number=3/4.
OCR on page.number=4/5.
OCR on page.number=5/6.
OCR on page.number=6/7.
OCR on page.number=7/8.
OCR on page.number=8/9.
OCR on page.number=9/10.
OCR on page.number=10/11.
OCR on page.number=11/12.
OCR on page.number=12/13.
OCR on page.number=13/14.
OCR on page.number=14/15.

Parsing file: ..\..\data\gemini_paper.pdf
=== Document parser messages ===
                                                                                                                                                                                                                                                                                                                                                                                                                       Using Tesseract for OCR processi

In [ ]:
model = BGEM3FlagModel('BAAI/bge-m3',  
                       use_fp16=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation

sentences_1 = ["What is BGE M3?", "Defination of BM25"]
sentences_2 = ["BGE M3 is an embedding model supporting dense retrieval, lexical matching and multi-vector interaction.", 
               "BM25 is a bag-of-words retrieval function that ranks a set of documents based on the query terms appearing in each document"]

embeddings_1 = model.encode(sentences_1, 
                            batch_size=12, 
                            max_length=8192, # If you don't need such a long length, you can set a smaller value to speed up the encoding process.
                            )['dense_vecs']
embeddings_2 = model.encode(sentences_2)['dense_vecs']
similarity = embeddings_1 @ embeddings_2.T
print(similarity)
# [[0.6265, 0.3477], [0.3499, 0.678 ]]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

In [6]:
def get_dense_emb(text, model_name = "text-embedding-3-large"):

    client = AzureOpenAI(
        api_version="2024-12-01-preview",
        azure_endpoint=os.environ["AZURE_ENDPOINT"],
        api_key=os.environ["AZURE_API_KEY"]
    )

    response = client.embeddings.create(
        input=[text],
        model=model_name
    )

    return response.data[0].embedding

def get_bm25_emb(text):
    model = SparseTextEmbedding(model_name="Qdrant/bm25")
    return list(model.embed([text]))[0]


In [7]:
points = []

for i, text in enumerate(chunks):
    # 1. Generate Embeddings
    dense_vector = get_dense_emb(text)
    sparse_embedding = get_bm25_emb(text) # Returns a SparseVector object
    
    # 2. Map to Qdrant Point structure
    points.append(
        models.PointStruct(
            id=i,
            vector={
                "dense-vector": dense_vector,
                "sparse-bm25": models.SparseVector(
                    indices=sparse_embedding.indices.tolist(), 
                    values=sparse_embedding.values.tolist()
                )
            },
            payload={
                "text": text,
                # Add any other metadata here
            }
        )
    )


In [ ]:
# Initialize client (assuming collection is already created as shown previously)
client = QdrantClient(
    url=os.environ["QDRANT_ENDPOINT"],
    api_key=os.environ["QDRANT_API_KEY"]
)

# Assuming 'points' is your list of PointStructs
batch_size = 50 

for i in range(0, len(points), batch_size):
    batch = points[i : i + batch_size]
    client.upsert(
        collection_name="rag_database",
        points=batch
    )
    print(f"Uploaded batch {i // batch_size + 1}")


In [ ]:
query_text = "What can you tell me about Attention?"

# 1. Generate query embeddings using your functions
dense_query = get_dense_emb(query_text)
sparse_query_obj = get_bm25_emb(query_text)

# 2. Execute Hybrid Search (RRF)
results = client.query_points(
    collection_name="rag_database",
    prefetch=[
        # Dense Search
        models.Prefetch(
            query=dense_query, 
            using="dense-vector", 
            limit=20
        ),
        # Sparse BM25 Search
        models.Prefetch(
            query=models.SparseVector(
                indices=sparse_query_obj.indices.tolist(), 
                values=sparse_query_obj.values.tolist()
            ), 
            using="sparse-bm25", 
            limit=20
        ),
    ],
    # Combine results using Reciprocal Rank Fusion
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=10
)

# Display results
for point in results.points:
    print(f"ID: {point.id}, Score: {point.score}, Text: {point.payload['text'][:100]}...")
